# Receive

In [4]:
sequenceDiagram
    participant Client
    participant Server
    participant Libsodium

    rect rgba(240,240,255,0.6)
    Client->>Server: send 32-byte serverPK
    end

    Note over Server,Libsodium: Keypair generation
    Server->>Libsodium: crypto_kx_keypair(&clientPK, &clientSK, &clientSK)
    Libsodium-->>Server: clientPK (32 bytes), clientSK (32 bytes)

    Server->>Client: send 32-byte clientPK

    Note over Server,Libsodium: Derive session keys (server role)
    Server->>Libsodium: crypto_kx_server_session_keys( \
        &clientRX, &clientTX, \
        &clientPK, &clientSK, &serverPK )
    Libsodium-->>Server: clientRX (nonce material, 32 bytes), clientTX (session key, 32 bytes)

    rect rgba(255,240,240,0.6)
    Client->>Server: send 8-byte ciphertext (encrypted file length)
    end

    Note over Server,Libsodium: Decrypt file length
    Server->>Libsodium: crypto_stream_xchacha20_xor_ic( \
        ciphertext =&ciphertextFileLength, \
        plaintext =&plaintext, \
        length = 8 (0x08), \
        nonce =&clientRX, \
        ic =&blockCounterICsendData (initially 0), \
        key =&clientTX )
    Libsodium-->>Server: plaintext = file length (uint64)
    Note over Server: ICsendData += ceil(8/64)=1 → now 1

    Client->>Server: send ciphertextFileData (ciphertextFileLength bytes)

    Note over Server,Libsodium: Decrypt file data
    Server->>Libsodium: crypto_stream_xchacha20_xor_ic( \
        ciphertext = ciphertextFileData, \
        plaintext = plaintextFileData, \
        length = ciphertextFileLength, \
        nonce =&clientRX, \
        ic =&blockCounterICsendData (currently 1), \
        key =&clientTX )
    Libsodium-->>Server: plaintextFileData
    Note over Server: ICsendData += ceil(ciphertextFileLength/64)

    Note over Server: Write plaintextFileData → output_file

    Note over Server,Libsodium: Encrypt file to send back (separate IC)
    Server->>Libsodium: crypto_stream_xchacha20_xor_ic( \
        ciphertext = plaintextFileData, \
        plaintext = ciphertextFileData, \
        length = ciphertextFileLength, \
        nonce =&clientRX, \
        ic =&blockCounterICfileLength (initially 0), \
        key =&clientTX )
    Libsodium-->>Server: ciphertext (to send back)
    Note over Server: ICfileLength += ceil(ciphertextFileLength/64)

    Server->>Client: send encrypted file (ciphertextFileLength bytes)

    Client->>Server: send 1-byte ciphertext (transfer status)

    Note over Server,Libsodium: Decrypt 1-byte transfer status
    Server->>Libsodium: crypto_stream_xchacha20_xor_ic( \
        ciphertext =&transferStatus, \
        plaintext =&plaintextFileLength, \
        length = 1 (0x01), \
        nonce =&clientRX, \
        ic =&blockCounterICsendData (after file data), \
        key =&clientTX )
    Libsodium-->>Server: transferStatus (0 == ok)
    Note over Server: ICsendData += ceil(1/64)=1

# Send

In [5]:
sequenceDiagram
    participant Client
    participant Server
    participant Libsodium

    Client->>Libsodium: crypto_kx_keypair(&inputBuffer, &var_98, &var_98)   %% 0x20 (32 bytes pub/priv)
    Libsodium-->>Client: inputBuffer (32 bytes), var_98 (32 bytes)

    Client->>Server: send 32-byte inputBuffer (client public key)

    Server->>Client: send 32-byte serverPK

    Note over Client,Libsodium: derive session keys (client role)
    Client->>Libsodium: crypto_kx_client_session_keys( \
        &key (session TX, 32 bytes), &nonce (session RX, 32 bytes), \
        &inputBuffer, &var_98, &resultBuffer )
    Libsodium-->>Client: key, nonce

    Note over Server,Libsodium: derive session keys (server role)
    Server->>Libsodium: crypto_kx_server_session_keys( \
        &clientRX (nonce), &clientTX (key), \
        &clientPK, &clientSK, &serverPK )
    Libsodium-->>Server: clientRX (nonce), clientTX (key)

    %% ---- client sends file length (8 bytes) ----
    Note over Client: blockCounterIC (client) = 0
    Client->>Libsodium: crypto_stream_xchacha20_xor_ic( \
        ciphertext =&var_c0, plaintext =&plaintext_1, length = 8 (0x08), \
        nonce =&nonce, ic =&blockCounterIC (0), key =&key )
    Libsodium-->>Client: var_c0 (8 bytes ciphertext)
    Note over Client: blockCounterIC += ceil(8/64) = +1 → blockCounterIC == 1
    Client->>Server: send 8-byte var_c0

    %% ---- server decrypts file length (8 bytes) ----
    Note over Server: blockCounterICsendData = 0
    Server->>Libsodium: crypto_stream_xchacha20_xor_ic( \
        ciphertext =&ciphertextFileLength, plaintext =&plaintext, length = 8 (0x08), \
        nonce =&clientRX, ic =&blockCounterICsendData (0), key =&clientTX )
    Libsodium-->>Server: plaintext = file length (uint64)
    Note over Server: blockCounterICsendData += ceil(8/64) = +1 → blockCounterICsendData == 1
    Server->>Client: (none, implicit progress)

    %% ---- client sends file data (plaintext_2 bytes) ----
    Note over Client: plaintext_2 = file size (ftell result)
    Client->>Libsodium: crypto_stream_xchacha20_xor_ic( \
        ciphertext = rax_21, plaintext = BUFFER, length = plaintext_2, \
        nonce =&nonce, ic =&blockCounterIC (currently 1), key =&key )
    Libsodium-->>Client: rax_21 (ciphertext, plaintext_2 bytes)
    Note over Client: blockCounterIC += ceil(plaintext_2/64) → blockCounterIC == 1 + ceil(plaintext_2/64)
    Client->>Server: send plaintext_2 bytes (rax_21)

    %% ---- server receives file data and decrypts ----
    Note over Server: blockCounterICsendData == 1 (from length step)
    Server->>Libsodium: crypto_stream_xchacha20_xor_ic( \
        ciphertext = ciphertextFileData, plaintext = plaintextFileData, \
        length = ciphertextFileLength (== plaintext_2), \
        nonce =&clientRX, ic =&blockCounterICsendData (1), key =&clientTX )
    Libsodium-->>Server: plaintextFileData
    Note over Server: blockCounterICsendData += ceil(ciphertextFileLength/64) \
        → blockCounterICsendData == 1 + ceil(ciphertextFileLength/64)

    %% ---- server writes to disk and then encrypts file to send back (separate IC) ----
    Note over Server: blockCounterICfileLength = 0
    Server->>Libsodium: crypto_stream_xchacha20_xor_ic( \
        ciphertext = plaintextFileData (outgoing ciphertext buffer), \
        plaintext = ciphertextFileData (on-disk bytes), \
        length = ciphertextFileLength, \
        nonce =&clientRX, ic =&blockCounterICfileLength (0), key =&clientTX )
    Libsodium-->>Server: outgoing ciphertext (ciphertextFileLength bytes)
    Note over Server: blockCounterICfileLength += ceil(ciphertextFileLength/64) \
        → blockCounterICfileLength == ceil(ciphertextFileLength/64)
    Server->>Client: send ciphertextFileLength bytes

    %% ---- client receives returned ciphertext and decrypts with blockCounterIC_1 ----
    Note over Client: blockCounterIC_1 = 0
    Client->>Libsodium: read(...) receives ciphertext bytes into rax_25
    Client->>Libsodium: crypto_stream_xchacha20_xor_ic( \
        ciphertext = ciphertext, plaintext = rax_25, \
        length = plaintext_2, \
        nonce =&nonce, ic =&blockCounterIC_1 (0), key =&key )
    Libsodium-->>Client: ciphertext (decrypted to 'ciphertext' buffer)
    Note over Client: blockCounterIC_1 += ceil(plaintext_2/64) → blockCounterIC_1 == ceil(plaintext_2/64)

    %% ---- client compares plaintext and sends 1-byte status (using blockCounterIC) ----
    Note over Client: blockCounterIC currently == 1 + ceil(plaintext_2/64)
    Client->>Libsodium: crypto_stream_xchacha20_xor_ic( \
        ciphertext =&var_125, plaintext =&plaintext (1 byte), \
        length = 1 (0x01), \
        nonce =&nonce, ic =&blockCounterIC (currently 1 + ceil(plaintext_2/64)), key =&key )
    Libsodium-->>Client: var_125 (1 byte ciphertext)
    Note over Client: blockCounterIC += ceil(1/64) = +1 → final blockCounterIC == 2 + ceil(plaintext_2/64)
    Client->>Server: send 1-byte var_125

    %% ---- server decrypts the 1-byte status ----
    Server->>Libsodium: crypto_stream_xchacha20_xor_ic( \
        ciphertext =&transferStatus, plaintext =&plaintextFileLength, \
        length = 1 (0x01), \
        nonce =&clientRX, ic =&blockCounterICsendData (currently 1 + ceil(ciphertextFileLength/64)), key =&clientTX )
    Libsodium-->>Server: transferStatus (0 == ok)
    Note over Server: blockCounterICsendData += ceil(1/64) = +1 \
        → final blockCounterICsendData == 2 + ceil(ciphertextFileLength/64)
        